# Week 3 homework: Graph Transformer

Transformers are everywhere in modern machine learning, so it is natural to apply them to graphs. In this homework we will implement a Graph Transformer (GT). Note that there are two model flavours that are called Graph Transformers in the literature, and it is important not to confuse them. The first variant is a model where every pair of nodes in the graph can interact through attention. These version is suitable for small graphs such as molecules. Here attention does not take the graph structure into account it all, so for strong performance this structure needs to be encoded in some other way, e.g., with some form of graph-aware positional embeddings. But this is not the kind of Graph Transformers we will be talking about today. The other variant is a model where only nodes connected by an edge can interact through attention, i.e., each node only directly interacts with its neighbors (and longer interactions can be achieved by stacking multiple layers). This version is very scalable and can be used for large sparse graphs. This is the Graph Transformer we are going to implement.

Откройте ноутбук в Google Colab, выберите «Среда выполнения → Сменить среду
выполнения → GPU» и затем «Выполнить все». Нужен Python 3.10–3.12; первая
ячейка проверяет наличие NVIDIA GPU до установки больших пакетов. Если
она попросит перезапустить сеанс после замены уже загруженных библиотек,
выполните «Среда выполнения → Перезапустить сеанс» и снова «Выполнить все».
Начальная установка может занять несколько минут. CPU вместо GPU не используется.

Версии согласованы с [PyTorch 2.4.0 / CUDA 12.4](https://pytorch.org/get-started/previous-versions/#v240)
и [DGL 2.4.0](https://github.com/dmlc/dgl/releases/tag/v2.4.0);
CUDA-сборка DGL берётся из [официального репозитория колёс](https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html).
После установки отдельная ячейка проверяет CUDA и прямой/обратный проход DGL
на маленьком графе. В этой облачной среде GPU-проверка и полное обучение
не выполнялись; результаты обучения и значения метрик заранее не подставлены.

In [ ]:
# Colab GPU setup: inspect distribution metadata before importing ML packages.
import os
import subprocess
import sys
from importlib import metadata

os.environ["DGLBACKEND"] = "pytorch"
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(
        "Нужен Python 3.10–3.12 для выбранных сборок Torch/DGL. "
        f"Сейчас используется Python {sys.version_info.major}.{sys.version_info.minor}."
    )
if globals().get("_WEEK3_RESTART_REQUIRED", False):
    raise RuntimeError(
        "Библиотеки уже обновлены, но старые модули остаются в памяти. "
        "Перезапустите сеанс Colab и снова выполните все ячейки."
    )

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
        stderr=subprocess.STDOUT, text=True,
    ).strip()
except (OSError, subprocess.CalledProcessError) as error:
    raise RuntimeError(
        "NVIDIA GPU недоступен. Сначала выберите GPU в настройках Colab "
        "и повторите запуск; установка больших пакетов не начата."
    ) from error
if not gpu_info:
    raise RuntimeError("nvidia-smi не обнаружил GPU; выберите GPU в настройках Colab.")
print("Доступные GPU и версии драйвера:")
print(gpu_info)

def installed_version(distribution):
    try:
        return metadata.version(distribution)
    except metadata.PackageNotFoundError:
        return None

exact_versions = {
    "numpy": "1.26.4",
    "torch": "2.4.0+cu124",
    "torchvision": "0.19.0+cu124",
    "torchaudio": "2.4.0+cu124",
    "dgl": "2.4.0",
    "torch-geometric": "2.5.0",
}
extra_distributions = ["scikit-learn", "tqdm", "ipywidgets", "PyYAML", "pydantic", "pandas"]

def matches(distribution):
    actual = installed_version(distribution)
    if actual is None:
        return False
    if distribution == "dgl":
        # Official GPU wheels may encode CUDA in the local version suffix.
        return actual.split("+")[0] == exact_versions[distribution]
    if distribution in exact_versions:
        return actual == exact_versions[distribution]
    if distribution == "pydantic":
        return int(actual.split(".")[0]) >= 2
    return True

distribution_modules = {
    "numpy": "numpy", "torch": "torch", "torchvision": "torchvision",
    "torchaudio": "torchaudio", "dgl": "dgl", "torch-geometric": "torch_geometric",
}
loaded_replacements = [module for distribution, module in distribution_modules.items()
                       if module in sys.modules and not matches(distribution)]

def pip_install(*arguments):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", *arguments])

scientific = ["numpy", *extra_distributions]
if not all(matches(name) for name in scientific):
    pip_install("numpy==1.26.4", "scikit-learn", "tqdm", "ipywidgets",
                "PyYAML", "pydantic>=2", "pandas")
if not all(matches(name) for name in ("torch", "torchvision", "torchaudio")):
    pip_install("torch==2.4.0+cu124", "torchvision==0.19.0+cu124", "torchaudio==2.4.0+cu124",
                "--index-url", "https://download.pytorch.org/whl/cu124")
if not matches("dgl"):
    pip_install("dgl==2.4.0", "-f", "https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
if not matches("torch-geometric"):
    pip_install("torch-geometric==2.5.0")

required = [*exact_versions, *extra_distributions]
mismatches = {name: installed_version(name) for name in required if not matches(name)}
if mismatches:
    raise RuntimeError(f"После установки остаются несовместимые версии: {mismatches}")
if loaded_replacements:
    _WEEK3_RESTART_REQUIRED = True
    raise RuntimeError(
        "Зависимости установлены, но заменённые модули уже были загружены: "
        + ", ".join(loaded_replacements)
        + ". Перезапустите сеанс Colab, затем снова выполните все ячейки."
    )
print("Нужные версии установлены; совпадающие пакеты повторно не устанавливаются.")

In [ ]:
# Fail before full-graph experiments if the GPU stack is not functional.
import os
os.environ["DGLBACKEND"] = "pytorch"
import numpy
import torch
import dgl
import torch_geometric

if not torch.cuda.is_available():
    raise RuntimeError(
        "Torch не видит CUDA GPU. Выберите GPU в Colab; если пакеты только "
        "обновлены, перезапустите сеанс и снова выполните все ячейки."
    )
if torch.version.cuda != "12.4":
    raise RuntimeError(f"Ожидалась сборка Torch CUDA 12.4, получена {torch.version.cuda}.")

tiny_graph = dgl.graph(([0, 1, 1], [1, 0, 1]), num_nodes=2, device="cuda:0")
tiny_features = torch.arange(6, dtype=torch.float32, device="cuda:0").reshape(2, 3).requires_grad_()
tiny_output = dgl.ops.copy_u_sum(tiny_graph, tiny_features)
tiny_expected = torch.stack([tiny_features[1], tiny_features[0] + tiny_features[1]])
torch.testing.assert_close(tiny_output, tiny_expected)
tiny_output.square().sum().backward()
if tiny_features.grad is None or not torch.isfinite(tiny_features.grad).all():
    raise RuntimeError("Проверка обратного прохода DGL на GPU не пройдена.")
torch.cuda.synchronize()
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("DGL:", dgl.__version__, "PyG:", torch_geometric.__version__, "NumPy:", numpy.__version__)
print("GPU-проверка DGL: прямой и обратный проход выполнены успешно.")
del tiny_graph, tiny_features, tiny_output, tiny_expected

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

First, let's get a graph. This is the same Amazon-Computers co-purchasing network we used in the seminar, so you are already familiar with it.

In [ ]:
from torch_geometric import datasets

In [ ]:
data = datasets.Amazon(name='computers', root='data')[0]
features = data.x
labels = data.y
edges = data.edge_index.T

print(f'Number of nodes: {len(labels)}')
print(f'Number of edges: {len(edges) // 2}')   # The graph is undirected, but is stored as a directed one,
                                               # so each edge appears twice.
print(f'Average node degree: {len(edges) / len(labels):.2f}')

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
full_idx = np.arange(len(labels))
train_idx, val_and_test_idx = train_test_split(full_idx, test_size=0.5, random_state=0,
                                               stratify=labels)

val_idx, test_idx = train_test_split(val_and_test_idx, test_size=0.5, random_state=0,
                                     stratify=labels[val_and_test_idx])

train_idx = torch.from_numpy(train_idx)
val_idx = torch.from_numpy(val_idx)
test_idx = torch.from_numpy(test_idx)

We will be using DGL in this homework.

In [ ]:
import dgl
from dgl import ops

In [ ]:
graph = dgl.graph((edges[:, 0], edges[:, 1]), num_nodes=len(labels))
graph

We want to allow each node to look not only at its neighbors but also at itself in the attention module, and a simple way to do it is to add self-loops to each node. Note that in the seminar we used an alternative approach where each node aggregated messages only from its neighbors, but then each aggregated neighborhood representation was concatenated with the representation of the ego-node. The same can be done in Graph Transformers, but today we will use the approach with self-loops, just so you are familiar with both.

In [ ]:
graph = dgl.add_self_loop(graph)
graph

A high-level overview of the Transformer block is shown in the image below (source: [CogView paper](https://arxiv.org/abs/2105.13290)). This is the pre-LayerNorm version of Transformer, which is more stable to train than the original post-LayerNorm version, and thus is used in most modern Transformer implementations.

![transformer-block](https://lh3.googleusercontent.com/d/1dvTlV4ZzHaCu-PXY_PtOJttMOi4h7N1t)

Let's start with the simple part: the Feed-Forward Network (FFN). This is just a two-layer MLP:

inputs -> linear layer -> dropout -> activation -> linear layer -> dropout -> outputs

Note that in the original implementation of Transformer the intermediate dimension of the FFN is 4 times larger than its output dimension, i.e., the inputs and outputs of the first linear layer have dimensions $dim$ and $4 \cdot dim$ respectively, while the inputs and outputs of the second linear layer have dimensions $4 \cdot dim$ and $dim$ respectively. The most currently popular hypothesis for why this helps is that [the feed-forward modules act as key-value memories](https://arxiv.org/abs/2012.14913) and thus need a large intermediate dimension to store a lot of patterns. However, graph node classification datasets are typically small and there are not so many patterns to memorize compared to language or images, so it is not clear if this dimension increase is needed for Graph Transformers. Your implementation should support various options for the intermediate dimension of the FFN (specifically, this dimension should be equal to `dim * dim_multiplier`).

Implement the Feed-Forward Network in the cell below **(1 pt)**. You can do it, we believe in you!

In [ ]:
class FeedForwardModule(nn.Module):
    def __init__(self, dim, dim_multiplier=4, dropout=0.2):
        super().__init__()

        hidden_dim = int(dim * dim_multiplier)
        self.layers = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.Dropout(dropout),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = self.layers(x)
        return x

Преобразования соответствуют схеме из условия:

$$\operatorname{FFN}(x)=\operatorname{Dropout}\!\left(W_2\,\operatorname{GELU}\!\left(\operatorname{Dropout}(W_1x+b_1)\right)+b_2\right).$$

Размерности: $W_1: d\to d\cdot m$, $W_2: d\cdot m\to d$, где $m=\texttt{dim\_multiplier}$. Нелинейность GELU выбрана как стандартная для Transformer; выход имеет размерность входа, что необходимо для residual connection.

In [ ]:
# A simple module initialization and output shape test.
feed_forward_module = FeedForwardModule(dim=256, dim_multiplier=4, dropout=0.2)
inputs = torch.randn(len(labels), 256)
outputs = feed_forward_module(inputs)
assert inputs.shape == outputs.shape

Now, let's implement the attention module **(3 pt)**. It is a bit more complicated. We provide the equations below. $x_i$ is the representation of node $v_i$ that is the input to the attention module, and $x_i^{out}$ is the representations of the corresponding node that the module outputs. For readability, we drop the head indices, but do not forget that the attention should be multi-headed.

$$ x_i \in \mathbb{R}^{model\_dim} $$

$$ query_i = W_{query} x_i + b_{query}, \;\;\;\;\; query_i \in \mathbb{R}^{head\_dim} $$

$$ key_i = W_{key} x_i + b_{key}, \;\;\;\;\; kay_i \in \mathbb{R}^{head\_dim} $$

$$ value_i = W_{value} x_i + b_{value}, \;\;\;\;\; value_i \in \mathbb{R}^{head\_dim} $$

The above operations are just standard linear layers.

Now, let's compute attention probabilities.

<br/>
<br/>

$$ s_{ij} = \frac{query_i^T key_j}{\sqrt{head\_dim}} \;\;\;\;\; \forall (i, j) \in E_{dir}, \;\;\;\;\; s_{ij} \in \mathbb{R}$$

$$ \Big( p_{ij}: \; v_j \in N(v_i) \Big) = softmax \Big( s_{ij}: \; v_j \in N(v_i) \Big) \;\;\;\;\; \forall i = 1, ..., n$$

This clunky notation means that for each node we take the softmax of attention scores ($s_{ij}$) of its neighbors to get attention probabilities ($p_{ij}$) corresponding to these neighbors. Another way to write it is:

$$ p_{ij} = \frac{ \exp{(s_{ij})} }{ \sum_{v_k \in N(v_i)} \exp{(s_{ik})} } \;\;\;\;\; \forall (i, j) \in E_{dir}, \;\;\;\;\; p_{ij} \in \mathbb{R} $$

The necessary edge softmax function is available in DGL.

It's time to aggregate neighborhood node representations based on the attention probabilities.

$$ x_i^{attn} = \sum_{v_j \in N(v_i)} p_{ij} value_j, \;\;\;\;\; x_i^{attn} \in \mathbb{R}^{head\_dim} $$

<br/>
<br/>

$$ x_i^{attn\_concat} = concatenate \Big( x_i^{attn} \; for \; all \; heads \Big), \;\;\;\;\; x_i^{attn\_concat} \in \mathbb{R}^{model\_dim} $$

And let's add one more linear layer with dropout.

$$ x_i^{out} = dropout \Big( W_{out} x_i^{attn\_concat} + b_{out} \Big), \;\;\;\;\; x_i^{out} \in \mathbb{R}^{model\_dim} $$

Note that your implementation should be efficient. To achieve this, you should use either `dgl.ops` or `dgl.function`. Inefficient implementations will not get any points.

If you struggle with implementing the attention module, review the GAT implementation from the seminar - there are some similar operations.

In [ ]:
from dgl.nn.functional import edge_softmax

In [ ]:
class DGLAttentionModule(nn.Module):
    def __init__(self, dim, num_heads, dropout):
        super().__init__()

        assert dim % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.query_proj = nn.Linear(dim, dim)
        self.key_proj = nn.Linear(dim, dim)
        self.value_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, graph, x):
        num_nodes = x.shape[0]

        # DGL: a directed edge (u, v) carries value from u to v.
        queries = self.query_proj(x).reshape(num_nodes, self.num_heads, self.head_dim)
        keys = self.key_proj(x).reshape(num_nodes, self.num_heads, self.head_dim)
        values = self.value_proj(x).reshape(num_nodes, self.num_heads, self.head_dim)

        # scores[(u, v), h] = dot(key[u, h], query[v, h]) / sqrt(head_dim).
        scores = ops.u_dot_v(graph, keys, queries) / (self.head_dim ** 0.5)
        attention = edge_softmax(graph, scores)  # Normalized per destination node.

        # Fused sparse aggregation: avoids constructing an [E, H, head_dim] tensor.
        x = ops.u_mul_e_sum(graph, values, attention)
        x = x.reshape(num_nodes, self.num_heads * self.head_dim)
        x = self.dropout(self.out_proj(x))
        return x

Для ребра $j\to i$ (источник $j$, получатель $i$) и каждой головы $h$ вычисляются

$$s_{ij}^{(h)}=\frac{\langle q_i^{(h)},k_j^{(h)}\rangle}{\sqrt{d_h}},\qquad
\alpha_{ij}^{(h)}=\frac{e^{s_{ij}^{(h)}}}{\sum_{k\in N(i)}e^{s_{ik}^{(h)}}},\qquad
z_i^{(h)}=\sum_{j\in N(i)}\alpha_{ij}^{(h)}v_j^{(h)}.$$

`ops.u_dot_v` считает скалярные произведения для существующих рёбер, `edge_softmax` нормирует их по входящим рёбрам узла, а `ops.u_mul_e_sum` выполняет *fused* взвешенную агрегацию. Поэтому не требуется создавать плотный тензор сообщений размера $|E|\times d$.

In [ ]:
# A simple module initialization and output shape test.
attention_module = DGLAttentionModule(dim=256, num_heads=4, dropout=0.2)
inputs = torch.randn(len(labels), 256)
outputs = attention_module(graph, inputs)
assert inputs.shape == outputs.shape

Now it is time to put everything together. Implement a Transformer block by combining AttentionModule, FeedForwardModule, LayerNorm layers, and skip connections (see the image in the beginning of the notebook) **(1 pt)**. Note that the `__init__` method of our TransformerModule class has an attention_module argument. For now, assume that it is going to be the DGLAttentionModule that you have implemented above. You will see why we did it this way a bit later.

In [ ]:
class TransformerModule(nn.Module):
    def __init__(self, attention_module, dim, num_heads, dim_multiplier, dropout):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attention = attention_module(dim=dim, num_heads=num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.feed_forward = FeedForwardModule(dim, dim_multiplier=dim_multiplier, dropout=dropout)

    def forward(self, graph, x):
        # Pre-LayerNorm + residual connections.
        x = x + self.attention(graph, self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x

Используем *Pre-LayerNorm*: сначала нормализация, затем attention с остаточным соединением, далее вторая нормализация и FFN с остаточным соединением:

$$h=x+\operatorname{Attention}(G,\operatorname{LN}_1(x)),\qquad
x_{out}=h+\operatorname{FFN}(\operatorname{LN}_2(h)).$$

In [ ]:
# A simple module initialization and output shape test.
transformer_module = TransformerModule(attention_module=DGLAttentionModule, dim=256, num_heads=4,
                                       dim_multiplier=4, dropout=0.2)
inputs = torch.randn(len(labels), 256)
outputs = transformer_module(graph, inputs)
assert inputs.shape == outputs.shape

Finally, implement the full model that consists of several Transformer blocks **(1 pt)**. The `num_layers` argument is the number of Transformer blocks. It is recommended to apply one more LayerNorm to the final output of the sequence of Transformer blocks. You also need some input transformation module to transform the dimension from the number of input features (which can be either too large or too small) to the model dimension, and an output head that transforms output representations (the ones after the final LayerNorm) to logits that should have dimension (num_nodes,) if there are only two classes and dimension (num_nodes, num_classes) otherwise.

In [ ]:
class GraphTransformer(nn.Module):
    def __init__(self, attention_module, num_layers, input_dim, hidden_dim, num_heads, hidden_dim_multiplier,
                 num_classes, dropout):
        super().__init__()

        self.num_classes = num_classes
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([
            TransformerModule(attention_module, dim=hidden_dim, num_heads=num_heads,
                              dim_multiplier=hidden_dim_multiplier, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1 if num_classes == 2 else num_classes)

    def forward(self, graph, x):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(graph, x)
        x = self.final_norm(x)
        logits = self.classifier(x)
        if self.num_classes == 2:
            logits = logits.squeeze(-1)
        return logits

Линейная проекция приводит признаки к `hidden_dim`; затем следуют `num_layers` блоков Transformer, финальная LayerNorm и классификационная голова. Для двух классов используется один логит на узел (для `BCEWithLogitsLoss`); для $C>2$ — $C$ логитов (для `CrossEntropyLoss`).

In [ ]:
# Simple module initialization and output shape tests.
input_dim = 1000
num_classes = 2
graph_transformer = GraphTransformer(attention_module=DGLAttentionModule, num_layers=2, input_dim=input_dim,
                                     hidden_dim=256, num_heads=4, hidden_dim_multiplier=4,
                                     num_classes=num_classes, dropout=0.2)
inputs = torch.randn(len(labels), input_dim)
outputs = graph_transformer(graph, inputs)
assert outputs.shape == (len(labels),)

input_dim = 100
num_classes = 10
graph_transformer = GraphTransformer(attention_module=DGLAttentionModule, num_layers=2, input_dim=input_dim,
                                     hidden_dim=256, num_heads=4, hidden_dim_multiplier=4,
                                     num_classes=num_classes, dropout=0.2)
inputs = torch.randn(len(labels), input_dim)
outputs = graph_transformer(graph, inputs)
assert outputs.shape == (len(labels), num_classes)

Now, imlement the code for running an experiment with Graph Transformer. It should create the model, prepare the data, train the model, and print the results. Your code should support both binary and multiclass classification: you should use different loss functions for these two cases, and report ROC AUC for binary classification and accuracy for multiclass classification. We also recommend to add AMP support, since it can make the training much faster and more memory efficient. Your function should return the trained model - you will se why we need it a bit later. Of course, you can just copy the code from the seminar, but we hope you will at least read it.

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
from torch.cuda.amp import autocast, GradScaler

In [ ]:
import time


def _classification_loss(logits, labels, indices, binary):
    if binary:
        return F.binary_cross_entropy_with_logits(logits[indices], labels[indices].float())
    return F.cross_entropy(logits[indices], labels[indices].long())


def _classification_metric(logits, labels, indices, binary):
    """ROC AUC for binary classification, accuracy otherwise."""
    target = labels[indices].detach().cpu().numpy()
    if binary:
        prob = torch.sigmoid(logits[indices].float()).detach().cpu().numpy()
        return roc_auc_score(target, prob) if len(np.unique(target)) == 2 else float('nan')
    pred = logits[indices].argmax(dim=-1).detach().cpu().numpy()
    return (pred == target).mean().item()


def run_experiment(graph, features, labels, train_idx, val_idx, test_idx, attention_module, num_layers=2,
                   hidden_dim=256, num_heads=4, hidden_dim_multiplier=4, dropout=0.2, lr=3e-5, num_steps=500,
                   device='cuda:0', amp=False):
    device = torch.device(device)
    if device.type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable; pass device="cpu" to run_experiment.')
    use_amp = bool(amp and device.type == 'cuda')

    # Full-batch transductive learning: the graph contains all nodes, but loss is
    # computed ONLY on train_idx. Validation chooses the best checkpoint.
    graph = graph.to(device)
    features = features.to(device=device, dtype=torch.float32)
    labels = labels.to(device).reshape(-1)
    train_idx = train_idx.to(device)
    val_idx = val_idx.to(device)
    test_idx = test_idx.to(device)

    num_classes = int(labels.unique().numel())
    binary = (num_classes == 2)
    model = GraphTransformer(
        attention_module=attention_module,
        num_layers=num_layers, input_dim=features.size(-1),
        hidden_dim=hidden_dim, num_heads=num_heads,
        hidden_dim_multiplier=hidden_dim_multiplier,
        num_classes=num_classes, dropout=dropout,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scaler = GradScaler(enabled=use_amp)

    metric_name = 'ROC AUC' if binary else 'accuracy'
    best_val_loss = float('inf')
    best_step = 0
    best_state = None
    started = time.perf_counter()

    for step in tqdm(range(1, num_steps + 1), desc=attention_module.__name__):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=use_amp):
            logits = model(graph, features)
            loss = _classification_loss(logits, labels, train_idx, binary)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Compute validation metrics periodically to avoid unnecessary extra
        # full-graph forward passes (especially expensive for PyG).
        if step == 1 or step % 25 == 0 or step == num_steps:
            model.eval()
            with torch.no_grad(), autocast(enabled=use_amp):
                val_logits = model(graph, features)
                val_loss = _classification_loss(val_logits, labels, val_idx, binary).item()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_step = step
                best_state = {key: value.detach().cpu().clone()
                              for key, value in model.state_dict().items()}
            if step == 1 or step % 100 == 0 or step == num_steps:
                val_metric = _classification_metric(val_logits, labels, val_idx, binary)
                print(f'Step {step:4d}/{num_steps}: train loss = {loss.item():.4f}; '
                      f'val loss = {val_loss:.4f}; val {metric_name} = {val_metric:.4f}')

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        # FP32 inference is consistent with the cross-framework parity checks.
        logits = model(graph, features)
        for split_name, indices in [('train', train_idx), ('validation', val_idx), ('test', test_idx)]:
            score = _classification_metric(logits, labels, indices, binary)
            print(f'{split_name.capitalize()} {metric_name}: {score:.4f}')

    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    print(f'Best checkpoint: step {best_step}, val loss {best_val_loss:.4f}; '
          f'elapsed: {time.perf_counter() - started:.1f} s')
    return model.cpu()

Обучение и метрики. Применяется full-batch transductive обучение: признаки и структура всего графа могут участвовать в message passing, но `loss` вычисляется только на `train_idx`. `val_idx` используется для выбора весов с минимальным validation loss, а `test_idx` — только для итоговой оценки после окончания обучения. Для бинарной задачи отчёт содержит ROC AUC, для многоклассовой — accuracy. AMP включается только при наличии GPU и запросе `amp=True`.

Параметры `num_steps` и `lr` оставлены как в задании. Время выполнения печатается для сравнения реализаций.

Let's see how our Graph Transformer works. You should get test accuracy above 0.895.

In [ ]:
graph_transformer_dgl = run_experiment(graph=graph, features=features, labels=labels,
                                       train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
                                       attention_module=DGLAttentionModule, device=device, amp=True)

As has been mentioned in the seminar, there is one more popular library for creating GNNs - [PyTorch Geometric (PyG)](https://github.com/pyg-team/pytorch_geometric). It is useful to be familiar with it as well, so now we will reimplement the Graph Transformer in PyG. Don't be afraid, all ypu need to do is create a new version of the attention module.

But first, we need a new graph representation. PyG does not have a dedicated graph class, but instead represents graphs as a (transposed) edge list tensor (i.e., a sparse adjacency matrix in COO format) which is typically called `edge_index`. Note that all graphs in PyG are considered directed (similar to DGL). Also, we should not forget that our implementation of Graph Transformer expects that each node has a self-loop.

In [ ]:
from torch_geometric.utils import add_self_loops

In [ ]:
edge_index = edges.T
edge_index, _ = add_self_loops(edge_index, num_nodes=len(features))
edge_index

Now, read the PyG GNN creation tutorial [here](https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_gnn.html) and try to adapt what you've learned from it to create the Graph Transformer attention module **(4 pts)**.

In [ ]:
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax

In [ ]:
class PyGAttentionModule(MessagePassing):
    def __init__(self, dim, num_heads, dropout):
        super().__init__(aggr='add', node_dim=0)

        assert dim % num_heads == 0

        # Names and shapes MUST match DGLAttentionModule for state_dict loading.
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.query_proj = nn.Linear(dim, dim)
        self.key_proj = nn.Linear(dim, dim)
        self.value_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, graph, x):
        # graph is edge_index with shape [2, num_edges]; j -> i.
        num_nodes = x.shape[0]
        queries = self.query_proj(x).reshape(num_nodes, self.num_heads, self.head_dim)
        keys = self.key_proj(x).reshape(num_nodes, self.num_heads, self.head_dim)
        values = self.value_proj(x).reshape(num_nodes, self.num_heads, self.head_dim)

        x = self.propagate(edge_index=graph, keys=keys, queries=queries, values=values)
        x = x.reshape(num_nodes, self.num_heads * self.head_dim)
        x = self.dropout(self.out_proj(x))
        return x

    def message(self, queries_i, keys_j, values_j, index):
        # i is the destination, j is the source.
        scores = (queries_i * keys_j).sum(dim=-1) / (self.head_dim ** 0.5)
        attention = softmax(scores, index=index)  # Softmax over incoming edges per head.
        messages_j = attention.unsqueeze(-1) * values_j
        return messages_j

`propagate()` передаёт в `message()` признаки отправителя с суффиксом `_j` и признаки получателя с суффиксом `_i`. Функция `softmax(scores, index)` нормирует оценки отдельно для входящих рёбер каждого целевого узла. Аггрегация `aggr='add'` суммирует сообщения по источникам.

В отличие от DGL, здесь формируется тензор сообщений размера $|E|\times H\times d_h$. Поскольку названия и формы параметров совпадают с DGL-реализацией, тесты с `load_state_dict()` сопоставляют две математически одинаковые модели.

In [ ]:
# A simple module initialization and output shape test.
attention_module = PyGAttentionModule(dim=256, num_heads=4, dropout=0.2)
inputs = torch.randn(len(labels), 256)
outputs = attention_module(edge_index, inputs)
assert inputs.shape == outputs.shape

In [ ]:
graph_transformer_pyg = GraphTransformer(
    attention_module=PyGAttentionModule,
    num_layers=2,
    input_dim=features.shape[1],
    hidden_dim=256,
    num_heads=4,
    hidden_dim_multiplier=4,
    num_classes=len(labels.unique()),
    dropout=0.2
)

In [ ]:
# A simple module initialization and output shape test.
outputs = graph_transformer_pyg(graph=edge_index, x=features)
assert outputs.shape == (len(labels), len(labels.unique()))

Let's test if your Graph Transformer implementations using two different libraries indeed compute the same thing.

In [ ]:
graph_transformer_dgl = graph_transformer_dgl.eval()
graph_transformer_pyg = graph_transformer_pyg.eval()

In [ ]:
state_dict = graph_transformer_dgl.state_dict()
graph_transformer_pyg.load_state_dict(state_dict)

In [ ]:
with torch.no_grad():
    logits_dgl = graph_transformer_dgl(graph=graph, x=features)
    logits_pyg = graph_transformer_pyg(graph=edge_index, x=features)
    
    assert torch.norm(logits_dgl - logits_pyg) < 1e-3


Now, let's look at one more dataset - Tolokers. In this graph, the nodes are workers from the Toloka crowdsourcing platform (tolokers), and they are connected by an edge if they have worked on the same task. The node features are based on the performance statistics and profile information of the workers. The labels indicate whether a worker was banned in one of the projects (binary classification task).

We will train both DGL Graph Transformer and PyG Graph Transformer on this dataset and compare their efficiency (both time spent and memory used).

In [ ]:
data = datasets.HeterophilousGraphDataset(name='tolokers', root='data/heterophilous-graphs')[0]
features = data.x
labels = data.y
edges = data.edge_index.T

# The graph is undirected, but is stored as a directed one (like all graphs in PyTorch Geometric),
# so each edge appears twice.
print(f'Number of nodes: {len(labels)}')
print(f'Number of edges: {len(edges) // 2}')
print(f'Average node degree: {len(edges) / len(labels):.2f}')
print(f'Number of classes: {len(labels.unique())}')

In [ ]:
train_idx = torch.where(data.train_mask[:, 0])[0]
val_idx = torch.where(data.val_mask[:, 0])[0]
test_idx = torch.where(data.test_mask[:, 0])[0]

In [ ]:
graph = dgl.graph((edges[:, 0], edges[:, 1]), num_nodes=len(labels))
graph = dgl.add_self_loop(graph)
graph

In [ ]:
edge_index = edges.T
edge_index, _ = add_self_loops(edge_index, num_nodes=len(features))
edge_index

In [ ]:
if torch.cuda.is_available() and str(device).startswith('cuda'):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

graph_transformer_dgl = run_experiment(graph=graph, features=features, labels=labels,
                                       train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
                                       attention_module=DGLAttentionModule, num_layers=3, hidden_dim=400,
                                       device=device, amp=True)

if torch.cuda.is_available() and str(device).startswith('cuda'):
    gpu_memory_used_gib = torch.cuda.max_memory_allocated(device) / 2 ** 30
    print(f'GPU memory used: {gpu_memory_used_gib:.2f} GiB')
else:
    print('GPU memory benchmark skipped: CUDA is unavailable')

In [ ]:
if torch.cuda.is_available() and str(device).startswith('cuda'):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

graph_transformer_pyg = run_experiment(graph=edge_index, features=features, labels=labels,
                                       train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
                                       attention_module=PyGAttentionModule, num_layers=3, hidden_dim=400,
                                       device=device, amp=True)

if torch.cuda.is_available() and str(device).startswith('cuda'):
    gpu_memory_used_gib = torch.cuda.max_memory_allocated(device) / 2 ** 30
    print(f'GPU memory used: {gpu_memory_used_gib:.2f} GiB')
else:
    print('GPU memory benchmark skipped: CUDA is unavailable')

You should get test ROC AUC above 0.79 for both models. Look at the time spent and GPU memory used numbers for both experiments. Now you know which library allows creating more efficient models. Think why there is such a big difference in efficiency between the two implementations of the Graph Transformer (hint: think about the sizes of tensors that get created during message passing).

The results of the two models above are slightly different due to initialization and training randomness (when trained on small graphs, GNNs tend to have a rather large variance between different training runs, so in real life projects it is always recommended to average model performance across several training runs). Let's verify once again that, given the same parameters, the two models compute the same thing.

### Почему DGL может использовать меньше памяти, чем PyG?

Реализация PyG создаёт по каждому ребру вектор `values_j` и затем тензор сообщений `[num_edges, num_heads, head_dim]`, то есть примерно $|E|\cdot d$ элементов (а при вычислении градиентов могут храниться дополнительные тензоры сопоставимого размера). При больших разреженных графах это дорого.

DGL использует `u_dot_v` и `u_mul_e_sum`: компактный тензор коэффициентов attention имеет порядок $|E|\cdot H$, а умножение значений на коэффициенты и суммирование выполняется оптимизированным разреженным оператором без материализации полного тензора сообщений размера $|E|\cdot d$. При $d\gg H$ экономия памяти потенциально значительная; разница времени зависит от устройства, версии библиотек и структуры графа.

Важно: точные значения ROC AUC, времени обучения и пикового расхода памяти можно получить только после выполнения обучающих ячеек на конкретной системе; они не подставлены заранее.

In [ ]:
graph_transformer_dgl = graph_transformer_dgl.eval()
graph_transformer_pyg = graph_transformer_pyg.eval()

In [ ]:
state_dict = graph_transformer_pyg.state_dict()
graph_transformer_dgl.load_state_dict(state_dict)

In [ ]:
with torch.no_grad():
    logits_dgl = graph_transformer_dgl(graph=graph, x=features)
    
    # In case you are using free Google Colab, this computation will not fit in RAM, so it is moved to GPU.
    graph_transformer_pyg = graph_transformer_pyg.to(device)
    logits_pyg = graph_transformer_pyg(graph=edge_index.to(device), x=features.to(device)).cpu()
    graph_transformer_pyg = graph_transformer_pyg.cpu()
    
    assert torch.norm(logits_dgl - logits_pyg) < 1e-3
